# Week 5 · Consultar traces de Logfire

Aquest notebook resol la pregunta 5: trobar el `trace_id` de la sessió agrupada sota `trivia_session` i contrastar-lo amb el dashboard.

Executa les cel·les després de recarregar Codespaces perquè el kernel disposi del secret `LOGFIRE_READ_TOKEN`. 

In [3]:


import os
from datetime import datetime
from zoneinfo import ZoneInfo

from IPython.display import display
from logfire.query_client import LogfireQueryClient


## 1. Configurar el client i el període

Les dates estan expressades en hora de Madrid. El període inicial cobreix el dia de la sessió de Q4; modifica’l si consultes sessions d’un altre dia. El client detecta la regió a partir del token.


In [5]:
madrid = ZoneInfo("Europe/Madrid")
min_timestamp = datetime(2026, 11, 1, tzinfo=madrid)
max_timestamp = datetime(2026, 11, 2, tzinfo=madrid)

# Aquesta cel·la no mostra el secret ni consulta encara el servidor.
client = LogfireQueryClient(read_token=os.environ["LOGFIRE_READ_TOKEN"])


## 2. Obtenir els identificadors de les sessions agrupades

Cada fila correspon al span pare d’una sessió. `kind = 'span'` selecciona spans completats. Les dates retornades poden estar en UTC; també les mostrem en hora de Madrid per comparar-les amb el dashboard.


In [15]:
sessions_sql = """
SELECT start_timestamp, trace_id, span_id, span_name
FROM records
WHERE span_name = 'trivia_session'
  AND kind = 'span'
ORDER BY start_timestamp DESC
LIMIT 100
"""

sessions_result = client.query_json_rows(
    sessions_sql,
    min_timestamp=min_timestamp,
    max_timestamp=max_timestamp,
)
sessions = sessions_result["rows"]

for index, session in enumerate(sessions):
    started = datetime.fromisoformat(session["start_timestamp"].replace("Z", "+00:00"))
    print(f"[{index}] {started.astimezone(madrid).isoformat()} | trace_id: {session['trace_id']}")

if not sessions:
    print("No s’han trobat sessions agrupades. Revisa el període i el projecte del read token.")


[0] 2026-11-01T18:32:10.424941+01:00 | trace_id: 01a1982b333807dfb0951da124116d81


## 3. Seleccionar la sessió 

- Selecciona `trivia_session` 
- Recupera els spans de la mateixa traça. Hi ha el pare `trivia_session`, els runs `invoke_agent agent` i les crides al model i a les eines.


In [11]:
session_index = 0  # Canvia’l si la sessió de Q4 no és la primera de la llista.
if not sessions:
    raise ValueError("Primer cal trobar una sessió amb la consulta anterior.")

trace_id = sessions[session_index]["trace_id"]
# Valida el valor abans d’incloure’l al SQL.
if len(trace_id) != 32 or any(char not in "0123456789abcdefABCDEF" for char in trace_id):
    raise ValueError("El trace_id ha de tenir 32 caràcters hexadecimals.")

trace_sql = f"""
SELECT start_timestamp, trace_id, span_id, parent_span_id, span_name
FROM records
WHERE trace_id = '{trace_id}'
  AND kind = 'span'
ORDER BY start_timestamp
LIMIT 1000
"""

trace_result = client.query_json_rows(
    trace_sql,
    min_timestamp=min_timestamp,
    max_timestamp=max_timestamp,
)
display(trace_result["rows"])
print("Trace ID seleccionat:", trace_id)
print("Runs de l’agent:", sum(row["span_name"] == "invoke_agent agent" for row in trace_result["rows"]))


[{'start_timestamp': '2026-11-01T17:32:10.424941Z',
  'trace_id': '01a1982b333807dfb0951da124116d81',
  'span_id': 'ccdbe3a927ffa0c3',
  'parent_span_id': None,
  'span_name': 'trivia_session'},
 {'start_timestamp': '2026-11-01T17:32:10.438012Z',
  'trace_id': '01a1982b333807dfb0951da124116d81',
  'span_id': 'e4d9e2aa6abf1485',
  'parent_span_id': 'ccdbe3a927ffa0c3',
  'span_name': 'invoke_agent agent'},
 {'start_timestamp': '2026-11-01T17:32:10.440590Z',
  'trace_id': '01a1982b333807dfb0951da124116d81',
  'span_id': '00cc242ff56aed5a',
  'parent_span_id': 'e4d9e2aa6abf1485',
  'span_name': 'chat gpt-4o-mini'},
 {'start_timestamp': '2026-11-01T17:32:12.083532Z',
  'trace_id': '01a1982b333807dfb0951da124116d81',
  'span_id': '2b324101208bf275',
  'parent_span_id': 'e4d9e2aa6abf1485',
  'span_name': 'execute_tool get_categories'},
 {'start_timestamp': '2026-11-01T17:32:12.460405Z',
  'trace_id': '01a1982b333807dfb0951da124116d81',
  'span_id': '4a44bfa5e8fa99e1',
  'parent_span_id': 'e4d

Trace ID seleccionat: 01a1982b333807dfb0951da124116d81
Runs de l’agent: 6


## 4. Contrastar amb el dashboard i entregar

[Projecte de Logfire](https://logfire-eu.pydantic.dev/a0010dev/starter-project), check `trivia_session`  **trace_id**  against the one obtained.


## 5. Q6 · Reconstruir tots els runs de la sessió

`trace_id` identifica tota la sessió agrupada. Cada span `invoke_agent agent` és un run diferent. Consultem **tots** els runs completats, sense el `LIMIT 1` de `fetch_trace()`.

Reconstruïm un `AgentRunResult` per run amb `TraceData` i `trace_to_run_result`. Cada run pot incloure l’historial acumulat: no concatenem els historials perquè duplicaríem missatges. Per al cost, sumem l’ús de cada run una sola vegada; no hi afegim els tokens dels spans fills `chat`, perquè ja estan inclosos en els totals del run.


La versió actual desa els totals del run a `gen_ai.aggregated_usage.*`; `COALESCE` també admet els camps antics `gen_ai.usage.*`. Selecciona el primer valor disponible: no suma les dues representacions.


In [18]:
from trace_replay import TraceData, trace_to_run_result

import json

runs_sql = f"""
SELECT
    start_timestamp,
    span_id,
    span_name,
    attributes->'pydantic_ai.all_messages' AS all_messages,
    COALESCE(
        attributes->>'gen_ai.aggregated_usage.input_tokens',
        attributes->>'gen_ai.usage.input_tokens'
    ) AS input_tokens,
    COALESCE(
        attributes->>'gen_ai.aggregated_usage.output_tokens',
        attributes->>'gen_ai.usage.output_tokens'
    ) AS output_tokens
FROM records
WHERE trace_id = '{trace_id}'
  AND kind = 'span'
  AND (span_name = 'agent run' OR span_name = 'invoke_agent agent')
ORDER BY start_timestamp ASC
"""

run_rows = client.query_json_rows(
    runs_sql,
    min_timestamp=min_timestamp,
    max_timestamp=max_timestamp,
)["rows"]

if not run_rows:
    raise ValueError("No s’han trobat runs de l’agent per al trace_id seleccionat.")

run_results = []
for row in run_rows:
    messages = row["all_messages"]
    if isinstance(messages, str):
        messages = json.loads(messages)
    if not isinstance(messages, list):
        raise ValueError(f"No hi ha missatges reconstruïbles al span {row['span_id']}.")
    if row["input_tokens"] is None or row["output_tokens"] is None:
        raise ValueError(f"Falten dades de tokens al span {row['span_id']}.")

    trace_data = TraceData(
        all_messages=messages,
        input_tokens=int(row["input_tokens"]),
        output_tokens=int(row["output_tokens"]),
    )
    run_results.append(trace_to_run_result(trace_data))

print("Runs reconstruïts:", len(run_results))
print("Resposta de l’últim run:")
print(run_results[-1].output)


Runs reconstruïts: 6
Resposta de l’últim run:
That's close, but the correct answer is **D) 299,792,458 m/s**.

### Interesting Fact:
The speed of light in a vacuum is a universal constant and plays a crucial role in the theory of relativity. It is not just a measurement but also sets the limit for how fast information can travel across the universe. In fact, the famous equation \(E=mc^2\) demonstrates the relationship between energy (E), mass (m), and the speed of light (c).

### Final Score:
You got 4 out of 5 questions correct! Well done! 🎉

Would you like to try another round or explore something else?


## 6. Cost total de la sessió

Sumem els tokens dels runs reconstruïts. Els preus de l’exercici són **0,15 USD / milió de tokens d’entrada** i **0,60 USD / milió de tokens de sortida**. Contrasta els totals amb el hover de `trivia_session` al dashboard.


In [23]:
MODEL_PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
}

def calculate_cost(model_name, input_tokens, output_tokens):
    prices = MODEL_PRICES[model_name.lower()]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    return input_cost + output_cost


    
total_input_tokens = sum(result.usage.input_tokens for result in run_results)
total_output_tokens = sum(result.usage.output_tokens for result in run_results)
total_cost = calculate_cost("gpt-4o-mini", total_input_tokens, total_output_tokens)

print("Trace ID:", trace_id)
print("Total input tokens:", total_input_tokens)
print("Total output tokens:", total_output_tokens)
print(f"Cost total aproximat: ${total_cost:.8f} USD")


Trace ID: 01a1982b333807dfb0951da124116d81
Total input tokens: 5692
Total output tokens: 816
Cost total aproximat: $0.00134340 USD


## 7. Q7 (Bonus) · Consultar el feedback

La consulta següent compta els esdeveniments `user_feedback` i els agrupa pel valor de l’atribut `feedback`: **`1` = positiu (👍)** i **`-1` = negatiu (👎)**. Si l’usuari tria `Skip`, no s’envia cap esdeveniment i no entra al recompte.

Filtrem per `kind = 'log'` perquè el feedback es registra amb `logfire.info()`. El període configurat a la cel·la cobreix del **2 de novembre de 2026 (inclòs) al 5 de novembre (exclòs)**, en hora de Madrid. No filtrem per `trace_id`: volem comptar el feedback de totes les sessions d’aquest període.

**Resposta conceptual de Q7:** els esdeveniments queden a la mateixa traça mitjançant `logfire.attach_context()` amb el context de la sessió. En l’aplicació de terminal, si `logfire.info("user_feedback", feedback=feedback)` s’executa dins del mateix bloc `with logfire.span("trivia_session")`, hereta directament aquest context.


In [7]:
feedback_sql = """
SELECT
    attributes->>'feedback' AS feedback,
    COUNT(*) AS total
FROM records
WHERE span_name = 'user_feedback'
  AND kind = 'log'
GROUP BY attributes->>'feedback'
ORDER BY feedback DESC
"""

with LogfireQueryClient(
    read_token=os.environ["LOGFIRE_READ_TOKEN"]
) as feedback_client:
    feedback_result = feedback_client.query_json_rows(
        feedback_sql,
        min_timestamp=datetime(2026, 11, 1, tzinfo=madrid),
        max_timestamp=datetime(2026, 11, 5, tzinfo=madrid),
    )

display(feedback_result["rows"])

[{'feedback': '1', 'total': 2}, {'feedback': '-1', 'total': 1}]

In [12]:
client.client.close()
